# 전체 그림 업로드 결과를 쉽게 확인하기

[프로젝트 추가] 이 노트북은 개인 JSON 보고서를 읽고 공개 그림을 확인하는 학습용입니다. DB 수정·Storage 업로드·임베딩·OpenAI 호출은 실행하지 않습니다.

그림 파일 업로드가 완료돼도 그림 설명과 주제 연결의 검토가 완료됐다는 뜻은 아닙니다. 모든 그림을 챗봇에서 자동으로 표시하지는 않습니다.

이번에는 main의 개인 폴더만 수정하기로 했으므로 관련 확인 노트북도 이 개인 폴더 안에 둡니다. 자세한 설명은 `full_image_storage_guide.md`를 참고합니다.

In [ ]:
from pathlib import Path
import json

# [프로젝트 추가] 실행 폴더가 달라도 상위 폴더에서 현재 팀 프로젝트 위치를 찾습니다.
project_root = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / 'src/car_search_rag/zzong_santafe_lag/full_image_storage.py').is_file()
)
# [프로젝트 추가] 저장된 결과만 읽습니다. 원격 DB나 모델에 접속하지 않습니다.
report_path = project_root / 'data/zzong_santafe_lag/reports/full_images_upload_summary_20261003.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
if report['phase'] != 'completed':
    raise ValueError('아직 완료된 보고서가 아닙니다. 진행 단계부터 확인하세요.')
result = report['result']
print('전체 그림:', result['image_count'])
print('이번 작업에서 추가:', result['task_new_objects_added'])
print('작업 시작 전 그림:', result['task_initial_storage_objects'])
print('원격 원본 확인:', result['verified_public_images'])
print('실패:', result['failure_count'])
# [프로젝트 추가] MB는 바이트를 1,000,000으로 나눈 값입니다.
for label, field in [('그림 파일', 'storage_bytes'), ('개인 DB', 'db_relation_bytes'), ('측정 합계', 'combined_measured_bytes')]:
    print(f'{label}: {result[field] / 1_000_000:.2f} MB ({result[field]:,}바이트)')
print('원문·벡터·설명·검토 상태 보존:', result['protected_personal_rows_unchanged'])

In [ ]:
# [프로젝트 추가] 보고서에는 실제 추출 이름·픽셀 크기·경로·파일 SHA-256이 남아 있습니다.
# DB의 원래 비어 있던 이름·크기는 이번 파일 업로드 과정에서 덮어쓰지 않았습니다.
page_number = 33
page_images = [item for item in result['files'] if item['pdf_page'] == page_number]
print(f'PDF {page_number}쪽 그림 수:', len(page_images))
for item in page_images:
    print(item['file_name'], item['pixel_size'], f"{item['bytes']:,}바이트")
    print('Storage 경로:', item['path'])
    print('공개 그림 주소:', item['public_url'])

In [ ]:
from IPython.display import Image, display

# [프로젝트 적용] 해당 페이지의 공개 파일을 화면에 표시합니다. 그림 설명을 생성하지 않습니다.
# 이 셀을 실행하면 공개 파일을 읽는 네트워크 요청은 발생하지만 유료 모델 호출은 없습니다.
for item in page_images:
    display(Image(url=item['public_url']))